# 03 - H4 Rebranding statt Buzzword-Inflation

**Generische Digital-Begriffe wachsen volumetrisch, nicht in Treffer-Häufigkeit und ein politisches Etikett wird komplett ausgetauscht, ohne dass sich der dahinterstehende Förderinhalt wesentlich ändert.**

Essay-Bezug: Akt 3 (Wie wird darüber geredet?)

## These und politischer Anschluss

**Original-These.** Begriffe rund um Künstliche Intelligenz (KI) und Cloud-Technologien wachsen im Untersuchungszeitraum schneller als das tatsächlich dahinterstehende Digital-Volumen, ein Indikator für „Buzzword-Inflation": Sprache läuft dem Geld voraus.

**Präzisierte These nach erster Sichtung.** Die Buzzword-Inflation in dieser zugespitzten Form lässt sich nicht belegen. Was sich stattdessen zeigt, ist ein **politisches Rebranding**: Ein Förderlabel verschwindet vollständig, ein neues erscheint bei vergleichbarem Volumen, die Hightech-Strategie (Merkel-Ära, CDU-geprägt) weicht der Zukunftsstrategie (Ampel-Koalition).

**Was wir prüfen.** Erstens, welche Schlagwörter des Zentrums für Europäische Wirtschaftsforschung (ZEW) den Datensatz prägen und wie sich ihre Treffer- und Volumenzahlen zwischen 2019 und 2024 entwickeln. Zweitens, ob Treffer-Wachstum und Volumen-Wachstum auseinanderlaufen, das wäre der Buzzword-Indikator. Drittens den Spezialfall Hightech-/Zukunftsstrategie als Beleg für Rebranding statt Inflation.

## Methodik dieses Drilldowns

**Daten und Abgrenzung.** ZEW-Schlagwörter stehen im `titel_text` in eckigen Klammern, teils mit geschweiften Sub-Markern. `extract_tags()` parst sie per Regex und normalisiert (lowercase, getrimmt). Wir zählen sowohl Treffer (Anzahl Titel mit Tag) als auch Volumen (`digi_soll_eng`-Summe) je Tag und Jahr, beide Größen zusammen ergeben den Buzzword-Test.

**Pre-Mortem.** Was würde die Rebranding-These töten? Wenn die Hightech-Strategie nicht verschwindet, sondern unter neuem Namen mit deutlich anderem Volumen weiterläuft, dann wäre es eher Umbenennung mit inhaltlicher Verschiebung als reines Label-Rebranding. Auch ein durchgängig schnelleres Treffer- als Volumen-Wachstum über die generischen Top-Tags würde die Buzzword-These stützen und unsere Gegenthese schwächen.

**Erwartung.** Generische Digital-Begriffe (Software, IT, Digitalisierung) wachsen eher im Volumen als in der Treffer-Zahl, Bestandstitel werden aufgestockt, nicht neue Titel mit demselben Label angelegt. Die Hightech-Strategie verschwindet vollständig aus dem Datensatz, die Zukunftsstrategie erscheint neu bei vergleichbarem Volumen.

In [1]:
"""03 - H4: Rebranding statt Buzzword-Inflation.

Befund: Die ursprueglich vermutete Buzzword-Inflation laesst sich datenseitig
nicht belegen. Stattdessen zeigt sich politisches Rebranding - vor allem das
vollstaendige Verschwinden der Hightech-Strategie (Merkel-Aera) und das
Erscheinen der Zukunftsstrategie (Ampel) bei vergleichbarem Volumen.

Drilldowns:
  1. ZEW-Schlagwoerter aus titel_text extrahieren (eckige Klammern)
  2. Treffer- und Volumen-Trends 2019 vs. 2024 je Schlagwort
  3. Wachstumsvergleich n vs. Volumen (Inflations-Indikator)

Outputs: figures/h4_rebranding_slope.png
"""

'03 - H4: Rebranding statt Buzzword-Inflation.\n\nBefund: Die ursprueglich vermutete Buzzword-Inflation laesst sich datenseitig\nnicht belegen. Stattdessen zeigt sich politisches Rebranding - vor allem das\nvollstaendige Verschwinden der Hightech-Strategie (Merkel-Aera) und das\nErscheinen der Zukunftsstrategie (Ampel) bei vergleichbarem Volumen.\n\nDrilldowns:\n  1. ZEW-Schlagwoerter aus titel_text extrahieren (eckige Klammern)\n  2. Treffer- und Volumen-Trends 2019 vs. 2024 je Schlagwort\n  3. Wachstumsvergleich n vs. Volumen (Inflations-Indikator)\n\nOutputs: figures/h4_rebranding_slope.png\n'

In [2]:
import sys
import re
from pathlib import Path
from collections import Counter
sys.path.insert(0, '..')  # repo root, relativ zum notebooks/-Verzeichnis

import pandas as pd
from src.load import load

df = load()  # nutzt Repo-Root-Default aus src.load


def clean_tag(s):
    """Tag normalisieren: geschweifte Sub-Marker raus, lowercase, trim."""
    s = re.sub(r'\{|\}', '', s).strip().lower()
    s = re.sub(r'\s+', ' ', s)
    return s


def extract_tags(text):
    if not isinstance(text, str):
        return []
    return [clean_tag(t) for t in re.findall(r'\[([^\]]+)\]', text)]

In [3]:
df['tags'] = df['titel_text'].apply(extract_tags)
print(f"Titel mit mindestens einem ZEW-Tag: "
      f"{df['tags'].apply(len).gt(0).sum():,} von {len(df):,}")

Titel mit mindestens einem ZEW-Tag: 3,784 von 21,358


**Befund.** Knapp ein Fünftel aller Haushaltstitel trägt mindestens ein ZEW-Schlagwort, eine ausreichend breite Basis für einen Trend-Vergleich auf Tag-Ebene, aber auch ein Hinweis darauf, dass die Mehrheit der digitalen Titel ohne explizites Themen-Label auskommt. Tags sind eine Zusatzinformation der ZEW-Klassifikation, keine Vollerhebung.

Nächster Schritt: Welche Schlagwörter dominieren den Datensatz?

In [4]:
all_tags = Counter()
for t in df['tags']:
    all_tags.update(t)

print("\n=== Top-20 ZEW-Schlagwoerter, alle Jahre kumuliert ===")
for tag, n in all_tags.most_common(20):
    print(f"  {n:>5}  {tag}")


=== Top-20 ZEW-Schlagwoerter, alle Jahre kumuliert ===
    911  software
    414  informationstechn
    197  it
    187  netzpolitik
    164  bundesnetzagentur
    158  e mobilität
    133  bundesamt für sicherheit in der informationstechnik
    124  bundeskartellamt
    105  digitale infrastruktur
     92  informationstechnikzentrum bund
     91  zukunftsstrategie
     89  zentrale stelle für informationstechnik im sicherheitsbereich
     86  datenschutz und die informationsfreiheit
     84  it-
     84  leibniz
     84  hightech-strategie
     75  digitale
     66  digitalisierung
     47  vernetz
     44  fernmelde


**Befund.** Die Top-Schlagwörter sind überwiegend Institutionen- und Funktionsbezeichnungen (Bundesnetzagentur, Bundesamt für Sicherheit in der Informationstechnik, Informationstechnikzentrum Bund) und generische IT-Begriffe (Software, Informationstechnik), keine Hype-Begriffe. „Zukunftsstrategie" und „Hightech-Strategie" tauchen mit vergleichbarer Häufigkeit auf — ein erster Hinweis auf den Label-Wechsel, der unten genauer untersucht wird.

Nächster Schritt: Wachsen Treffer-Zahl und Volumen dieser Tags zwischen 2019 und 2024 im Gleichschritt — oder läuft eines dem anderen voraus?

In [5]:
top_tags = [t for t, _ in all_tags.most_common(25)]
res = []
for tag in top_tags:
    has_tag = df['tags'].apply(lambda lst: tag in lst)
    row = {'tag': tag}
    for j in sorted(df['jahr'].unique()):
        sub = df[has_tag & (df['jahr'] == j)]
        row[f'n_{j}'] = len(sub)
        row[f'v_{j}_Mio'] = round(sub['digi_soll_eng'].sum() / 1e3, 1)
    res.append(row)
out = pd.DataFrame(res)

In [6]:
print("\n=== Wachstum Treffer vs. Volumen 2019 -> 2024 ===")
print(f"{'Tag':<35} {'n19':>4} {'n24':>4} {'n_x':>5} "
      f"{'v19_Mio':>9} {'v24_Mio':>9} {'v_x':>6}")
print("-" * 80)
for _, r in out.iterrows():
    n19, n24 = r['n_2019'], r['n_2024']
    v19, v24 = r['v_2019_Mio'], r['v_2024_Mio']
    n_fac = f"{n24/n19:.1f}x" if n19 > 0 else 'neu'
    v_fac = f"{v24/v19:.1f}x" if v19 > 0 else 'neu'
    print(f"{r['tag'][:34]:<35} {n19:>4} {n24:>4} {n_fac:>5} "
          f"{v19:>9.0f} {v24:>9.0f} {v_fac:>6}")


=== Wachstum Treffer vs. Volumen 2019 -> 2024 ===
Tag                                  n19  n24   n_x   v19_Mio   v24_Mio    v_x
--------------------------------------------------------------------------------
software                             214  237  1.1x       466       634   1.4x
informationstechn                     98  108  1.1x      1541      3539   2.3x
it                                    43   54  1.3x      1238      1052   0.8x
netzpolitik                           41   52  1.3x      1238      1037   0.8x
bundesnetzagentur                     37   45  1.2x        37        43   1.2x
e mobilität                           29   43  1.5x       748      1067   1.4x
bundesamt für sicherheit in der in    33   34  1.0x       137       238   1.7x
bundeskartellamt                      30   32  1.1x        11        22   2.1x
digitale infrastruktur                26   25  1.0x       218      3562  16.4x
informationstechnikzentrum bund       24   22  0.9x       535      1311   2.5x

**Befund.** Die ursprünglich vermutete Buzzword-Inflation, Treffer-Zahlen wachsen schneller als Volumen, lässt sich so nicht belegen. Bei den meisten generischen Begriffen (Software, Informationstechnik, digitale Infrastruktur, Digitalisierung) wächst das Volumen deutlich stärker als die Treffer-Zahl: Bestehende Titel werden finanziell aufgestockt, nicht neue Titel mit denselben Labels angelegt. Auffälligster Fall ist „Zukunftsstrategie": neu in der Tag-Liste, mit auf Anhieb 1,89 Mrd. €. Sein Gegenstück „Hightech-Strategie" verschwindet im selben Zeitraum vollständig, ein Etiketten-Tausch, kein organisches Wachstum.

Nächster Schritt: Dieser Tausch verdient einen genaueren Blick, wie verändern sich Treffer und Volumen von Hightech- und Zukunftsstrategie über alle vier Jahre?

[TODO: Stichproben-Vergleich einzelner Titel 2019/2024 hinsichtlich Bezeichnung und Gegenstand. Wie viele der 2019er Hightech-Titel finden sich 2024 wortgleich oder semantisch identisch unter Zukunftsstrategie wieder?]

In [7]:
#| label: tbl-rebranding
print("\n=== Rebranding-Befund: Politisches Etikett wechselt, Inhalt bleibt ===")
for tag in ['hightech-strategie', 'zukunftsstrategie']:
    has_tag = df['tags'].apply(lambda lst, t=tag: t in lst)
    for j in sorted(df['jahr'].unique()):
        sub = df[has_tag & (df['jahr'] == j)]
        n, v = len(sub), sub['digi_soll_eng'].sum() / 1e6
        print(f"  {tag:<22} {j}: n={n:>3}, V={v:>5.2f} Mrd")
    print()


=== Rebranding-Befund: Politisches Etikett wechselt, Inhalt bleibt ===
  hightech-strategie     2019: n= 40, V= 1.32 Mrd
  hightech-strategie     2021: n= 44, V= 1.49 Mrd
  hightech-strategie     2023: n=  0, V= 0.00 Mrd
  hightech-strategie     2024: n=  0, V= 0.00 Mrd

  zukunftsstrategie      2019: n=  0, V= 0.00 Mrd
  zukunftsstrategie      2021: n=  0, V= 0.00 Mrd
  zukunftsstrategie      2023: n= 45, V= 1.89 Mrd
  zukunftsstrategie      2024: n= 46, V= 1.89 Mrd



**Befund.** Der Befund ist eindeutig: Die Hightech-Strategie ist 2019/2021 mit 40–44 Titeln und rund 1,3–1,5 Mrd. € fest im Haushalt verankert, und 2023/2024 vollständig verschwunden, null Treffer, null Euro. Im selben Moment erscheint die Zukunftsstrategie, 2019/2021 noch nicht existent, 2023/2024 mit 45–46 Titeln und rund 1,89 Mrd. €. Die Größenordnung ist vergleichbar (Volumen-Faktor 1,43), das politische Etikett ist neu.

Das ist kein graduelles Aus- und Einlaufen, wie man es bei einem Politikwechsel erwarten würde, sondern ein abrupter, vollständiger Tausch, Ausdruck eines Regierungswechsels (Merkel- zu Ampel-Koalition), der sich im Förder-Vokabular niederschlägt, ohne dass sich das Volumen wesentlich ändert.

## Kernbefund und weiter

**Die vermutete Buzzword-Inflation lässt sich nicht belegen. Generische Digital-Begriffe wachsen volumetrisch, nicht in Treffer-Häufigkeit. Stattdessen zeigt sich ein politisches Rebranding: Die Hightech-Strategie verschwindet 2023/2024 vollständig (40 Treffer / 1,32 Mrd. € in 2019 auf null), die Zukunftsstrategie erscheint bei vergleichbarem Volumen neu (46 Treffer / 1,89 Mrd. € in 2024, Volumen-Faktor 1,43).** Sprache verändert sich an der Oberfläche stärker als das Fördervolumen darunter.

**Weiter zu Notebook 99** Synthese: Wie fügen sich Polyzentrik (H1), Transfer-Schere (H2) und Rebranding (H4) zur politischen Implikation zusammen, und was kann ein BMDS davon tatsächlich steuern?

**Limitationen dieser Analyse.** Die ZEW-Tags sind eine Zusatzklassifikation, keine Vollerhebung, knapp 18 Prozent der Titel tragen überhaupt ein Tag. Der Hightech-/Zukunftsstrategie-Befund beruht auf Aggregaten; ob es sich um dieselben Förderlinien unter neuem Namen handelt oder um inhaltlich verschobene Programme, lässt sich nur durch einen Stichproben-Vergleich auf Titel-Ebene klären (siehe TODO oben).